In [1]:
import sys
import os
sys.path.append("..") # 确保能导入 src
from datasets import Dataset, DatasetDict
import xml.etree.ElementTree as ET
import pandas as pd

In [2]:
# 1. 直接使用本地文件加载
def parse_semeval_xml(xml_path, domain):
    """
    手动解析 SemEval-2014 Task 4 的 XML 文件。
    
    Args:
        xml_path (str): XML 文件的本地路径。
        domain (str): "restaurants" 或 "laptops"。
        
    Returns:
        list: 包含每条评论数据的字典列表。
    """
    
    # 检查 domain 是否合法
    if domain not in ["restaurants", "laptops"]:
        raise ValueError("Domain must be either 'restaurants' or 'laptops'")

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except FileNotFoundError:
        print(f"错误: 找不到文件 {xml_path}")
        return []
    except ET.ParseError:
        print(f"错误: 无法解析 XML 文件 {xml_path}")
        return []

    data = []

    # 遍历 XML 中的每一个 <sentence> 标签
    for sentence in root.iter("sentence"):
        # 获取 sentence 的 id
        sentence_id = sentence.attrib.get("id")
        
        # 获取文本内容
        text_node = sentence.find("text")
        text = text_node.text if text_node is not None else ""
        
        # 提取 aspectTerms
        # 注意：Laptops 和 Restaurants 都有 aspectTerms
        aspect_terms = []
        # 使用 iter 查找当前句子下的所有 aspectTerm 标签
        for aspect_term in sentence.iter("aspectTerm"):
            # .attrib 会直接返回属性字典，例如 {'term': 'food', 'polarity': 'positive', ...}
            aspect_terms.append(aspect_term.attrib)
            
        # 构建基础数据项
        item = {
            "sentenceId": sentence_id,
            "text": text,
            "aspectTerms": aspect_terms
        }

        # 提取 aspectCategories (仅 Restaurants 有此字段)
        if domain == "restaurants":
            aspect_categories = []
            for aspect_category in sentence.iter("aspectCategory"):
                aspect_categories.append(aspect_category.attrib)
            item["aspectCategories"] = aspect_categories

        data.append(item)

    return data

# 2. 手动加载数据
def load_semeval_data(data_dir="../data/raw/SemEval2014Task4/"):
    """手动加载所有数据"""
    
    # 定义文件路径
    files = {
        "restaurants": {
            "trial": f"{data_dir}/restaurants-trial.xml",
            "train": f"{data_dir}/SemEval'14-ABSA-TrainData_v2 & AnnotationGuidelines/Restaurants_Train_v2.xml",
            "test": f"{data_dir}/ABSA_Gold_TestData/Restaurants_Test_Gold.xml"
        },
        "laptops": {
            "trial": f"{data_dir}/laptops-trial.xml",
            "train": f"{data_dir}/SemEval'14-ABSA-TrainData_v2 & AnnotationGuidelines/Laptop_Train_v2.xml",
            "test": f"{data_dir}/ABSA_Gold_TestData/Laptops_Test_Gold.xml"
        }
    }
    
    datasets = {}
    for domain in ["restaurants", "laptops"]:
        domain_data = {}
        for split in ["trial", "train", "test"]:
            try:
                samples = parse_semeval_xml(files[domain][split], domain)
                domain_data[split] = Dataset.from_list(samples)
                print(f"✅ 成功加载 {domain} {split}: {len(samples)} 样本")
            except Exception as e:
                print(f"❌ 加载 {domain} {split} 失败: {e}")
        
        datasets[domain] = DatasetDict(domain_data)
    
    return datasets

In [3]:
# 3. 使用
datasets = load_semeval_data()
ds_rest = datasets["restaurants"]
ds_lap = datasets["laptops"]

def print_check(ds):
    print("=== 数据集信息 ===")
    print(f"数据集结构: {ds}")
    print(f"训练集大小: {len(ds['train'])}")
    print(f"测试集大小: {len(ds['test'])}")
    print(f"试验集大小: {len(ds['trial'])}")

    df_train = pd.DataFrame(ds['train'].select(range(5)))
    print("前5个训练样本:")
    print(df_train[['text', 'aspectTerms']].to_string())

    print("\n=== 特征结构 ===")
    print(ds['train'].features)

    # 查看数据分布
    print("\n=== 情感极性分布 ===")
    from collections import Counter
    polarities = []
    for sample in ds['train']:
        for term in sample['aspectTerms']:
            polarities.append(term['polarity'])
    print(Counter(polarities))

print_check(ds_rest)
print_check(ds_lap)

✅ 成功加载 restaurants trial: 100 样本
✅ 成功加载 restaurants train: 3041 样本
✅ 成功加载 restaurants test: 800 样本
✅ 成功加载 laptops trial: 100 样本
✅ 成功加载 laptops train: 3045 样本
✅ 成功加载 laptops test: 800 样本
=== 数据集信息 ===
数据集结构: DatasetDict({
    trial: Dataset({
        features: ['sentenceId', 'text', 'aspectTerms', 'aspectCategories'],
        num_rows: 100
    })
    train: Dataset({
        features: ['sentenceId', 'text', 'aspectTerms', 'aspectCategories'],
        num_rows: 3041
    })
    test: Dataset({
        features: ['sentenceId', 'text', 'aspectTerms', 'aspectCategories'],
        num_rows: 800
    })
})
训练集大小: 3041
测试集大小: 800
试验集大小: 100
前5个训练样本:
                                                                                                                                                        text                                                                                                                                                                                                    

In [4]:
# 将 Dataset 转为 Pandas 以便查看和保存 CSV
df_rest_train = ds_rest['train'].to_pandas()
df_rest_test = ds_rest['test'].to_pandas()
df_rest_trail = ds_rest['trial'].to_pandas()

df_lap_train = ds_lap['train'].to_pandas()
df_lap_test = ds_lap['test'].to_pandas()
df_lap_trail = ds_lap['trial'].to_pandas()

os.makedirs("../data/processed", exist_ok=True)
df_rest_train.to_json("../data/processed/train_rest_clean.jsonl", orient='records', lines=True, force_ascii=False)
df_rest_test.to_json("../data/processed/test_rest_clean.jsonl", orient='records', lines=True, force_ascii=False)
df_rest_trail.to_json("../data/processed/trial_rest_clean.jsonl", orient='records', lines=True, force_ascii=False)

df_lap_train.to_json("../data/processed/train_lap_clean.jsonl", orient='records', lines=True, force_ascii=False)
df_lap_test.to_json("../data/processed/test_lap_clean.jsonl", orient='records', lines=True, force_ascii=False)
df_lap_trail.to_json("../data/processed/trial_lap_clean.jsonl", orient='records', lines=True, force_ascii=False)

print(f"Rest: Saved {len(df_rest_train)} training samples and {len(df_rest_test)} test samples and {len(df_rest_trail)} trail samples.")
print(f"Lap: Saved {len(df_lap_train)} training samples and {len(df_lap_test)} test samples and {len(df_lap_trail)} trail samples.")

Rest: Saved 3041 training samples and 800 test samples and 100 trail samples.
Lap: Saved 3045 training samples and 800 test samples and 100 trail samples.
